Model selection

In [1]:
# load libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as sm

netMHC1 default settings (peptide length 9, window 5, human 27 allele panel). Yielded 1 warning:\
2 duplicates found of input sequence: "QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLEWVAVIWYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDPRGATLYYYYYGMDVWGQGTTVTVSSDIQMTQSPSSLSASVGDRVTITCRASQSINSYLDWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQYYSTPFTFGPGTKVEIK"

In [ ]:
# load all data sets

# netMHC1 predictions
    # deafault settings (peptide length 9, window 5, human 27 allele panel)
    # separated into 6 predictions bc of issues with running full fasta file
netMHC1_EL_pep9_subset1 = pd.read_csv("tool_outputs/217AB_netMHC_peplen9_subset1.csv")

    # peptide length 14, window 5, human 27 allele panel
    # separated into 6 predictions bc of issues with running full fasta file
#netMHC1_EL_pep14_subset1 # not yet predicted

# netMHC II predicitons
    # default settings (peptide length 15, window 5, human 27 allele panel)
    # separated into 6 predictions bc of issues with running full fasta file
#netMHC_II_EL_pep15_subset1 # not yet predicted

    # peptide length 12, window 5, human 27 allele panel
    # separated into 6 predictions bc of issues with running full fasta file
#netMHC_II_EL_pep12_subset1 # not yet predicted

# sequence table from one of the netMHC predicitons. 
# The tools remove the antibody name and just put a number instead. This is so that the name can be mapped back later
# seperated into 6 predictions (with different antibodies in each) 
seqTable_subset1 = pd.read_csv("tool_outputs/seqTable217AB_netMHC_peplen9_subset1.csv")


# waltz predictions
# separated into 6 predicitons due to max capacity of waltz tool
waltz1 = pd.read_csv("tool_outputs/waltz1_217AB.txt")
waltz2 = pd.read_csv("tool_outputs/waltz2_217AB.txt")
waltz3 = pd.read_csv("tool_outputs/waltz3_217AB.txt")
waltz4 = pd.read_csv("tool_outputs/waltz4_217AB.txt")
waltz5 = pd.read_csv("tool_outputs/waltz5_217AB.txt")
waltz6 = pd.read_csv("tool_outputs/waltz6_217AB.txt")

# biophi predictions
    # default settings (numbering scheme Kabat, CDR definition Kabat, OASis prevalence threshold Relaxed)
    # seperated into 3 predicitons due to max capacity of biobhi humanness tool
biophi_Relaxed1 = pd.read_excel("tool_outputs/NRKab_CDRKab_Relaxed_pred1_217AB.xlsx")
biophi_Relaxed2 = pd.read_excel("tool_outputs/NRKab_CDRKab_Relaxed_pred2_217AB.xlsx")
biophi_Relaxed3 = pd.read_excel("tool_outputs/NRKab_CDRKab_Relaxed_pred3_217AB.xlsx")

    # numbering scheme Kabat, CDR definition Kabat, OASis prevalence threshold Strict
    # seperated into 3 predicitons due to max capacity of biobhi humanness tool
biophi_Strict1 = pd.read_excel("tool_outputs/NRKab_CDRKab_Strict_pred1_217AB.xlsx")
biophi_Strict2 = pd.read_excel("tool_outputs/NRKab_CDRKab_Strict_pred2_217AB.xlsx")
biophi_Strict3 = pd.read_excel("tool_outputs/NRKab_CDRKab_Strict_pred3_217AB.xlsx")


In [27]:
# load ADA scores
ADA = pd.read_csv("ADA_Clinical_Ab_2021_biophiDataset.csv")
# extract only the ADA (here called Immunogenicity) and Antibody (name) columns
ADA = ADA[["Antibody", "Immunogenicity"]].copy()
# make all antibody names lowercase
ADA["Antibody"] = ADA["Antibody"].str.lower()


# load another dataset with antibodies, that have been predicited previously
previous_37AB = pd.read_csv("../Antibodies/ADA_combined_features.csv")
# rename antibody column so it maches the one in the ADA df
previous_37AB = previous_37AB.rename(columns={'antibody': 'Antibody'})
# make all antibody names lowercase
previous_37AB["Antibody"] = previous_37AB["Antibody"].str.lower()


In [29]:
# Check how many/ which antibody names match in the ADA score table and in the previously predicted antibodies
# select antibody and ADA precentage column, rename so it matches ADA df
subset = previous_37AB[['Antibody', 'ADA_percentage']].rename(
    columns={'ADA_percentage': 'Immunogenicity'}
)

# collect the antibodies that do not exist in the ADA df
missing = subset[~subset['Antibody'].isin(ADA['Antibody'])]

# append to ADA df
ADA = pd.concat([ADA, missing], ignore_index=True)



In [ ]:
# Merge waltz dfs together

# Merge biophi Relaxed dfs together

# Merge biophi Strict dfs together

In [ ]:
# the netMHC tools do not have the antibody name in the output, only a sequence number
# Here the seq # will be mapped back to the original antibody name using the seqTable
netMHC1_EL_pep9 = netMHC1_EL_pep9.merge(seqTable[['seq #', 'sequence name']], how='left')

In [ ]:
# Sanity checks

# Check nr of antibodies
if netMHC1_EL_pep14['seq #'].nunique()!=217:
        print( "netMHC1_EL_pep14 does not have 217 antibodies")
if netMHC1_EL_pep9['seq #'].nunique()!=217:
        print( "netMHC1_EL_pep9 does not have 217 antibodies")
if netMHC_II_EL_pep12['seq #'].nunique()!=217:
        print( "netMHC_II_EL_pep12 does not have 217 antibodies")   
if netMHC_II_EL_pep15['seq #'].nunique()!=217:
        print( "netMHC_II_EL_pep15 does not have 217 antibodies")
if biophi_KabKabRelaxed['Antibody'].nunique()!=217:
        print( "biophi_KabKabRelaxed does not have 217 antibodies")
if biophi_KabKabStrict['Antibody'].nunique()!=217:
        print( "biophi_KabKabStrict does not have 217 antibodies")
if waltz['antibody'].nunique()!=217:
        print( "waltz does not have 217 antibodies")

# Compute scores
netMHCpan tools give one score for each peptide - HLA allele combination
Therefore, for each antibody a immunogenicity score will be calculated. The definition of what score is considerd immunogenetic differs for the different tools \
for each netMHC score of interest: precentile score, immunogenicity socre and pre processing score

In [ ]:
# netMHC1_EL_pep9 percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep9_percentile = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_percentile')
    )

# netMHC1_EL_pep9 Immunogenicity score 

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep9_immunogenicity_score = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_immunogenicity_score')
    )

# netMHC1_EL_pep9 Preprocessing score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMCH1_pep9_preProcess = netMHC1_EL_pep9.groupby('sequence name')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep9_preProcess'})

# netMHC1_EL_pep14

# Percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep14_percentile = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_percentile')
    )

# Immunogenicity score

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep14_immunogenicity_score = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_immunogenicity_score')
    )

# Pre-proocessing score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 

netMCH1_pep14_preProcess = netMHC1_EL_pep14.groupby('sequence name')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep14_preProcess'})


# netMHC_II_EL_pep12

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep12_percentile = (
    netMHC_II_EL_pep12.assign(immunogenic=netMHC_II_EL_pep12['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep12_percentile')
    )

# Immunogenicity score

# remove the rows with the immunogenicity score of '-' before calculating the mean
netMHC_II_EL_pep12 = netMHC_II_EL_pep12[netMHC_II_EL_pep12['immunogenicity score'] != '-']
# make the column with the immunogenicity score into a numeric column
netMHC_II_EL_pep12['immunogenicity score'] = pd.to_numeric(netMHC_II_EL_pep12['immunogenicity score'])

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody.
netMHC_II_pep12_immunogenicity_score = netMHC_II_EL_pep12.groupby('sequence name')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep12_immunogenicity_score'})


# Does not have pre processing score. Due to tool not being able to predict pre processing on peptides shorter than 13 amino acids


# netMHC_II_EL_pep15

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep15_percentile = (
    netMHC_II_EL_pep15.assign(immunogenic=netMHC_II_EL_pep15['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep15_percentile')
    )

# Immunogenicity score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_immunogenicity_score = netMHC_II_EL_pep15.groupby('sequence name')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep15_immunogenicity_score'})

# Pre-proocessing score
# MHC class 2 has 2 preprocessing scores of interest: mhcii-np cleavage probability score and mhcii-np cleavage probability percentile rank

# mhcii-np cleavage probability

# remove the rows with the cleavage probability score of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability score'] != '-']
# make the column with the cleavage probability score into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability score'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability score'])
# Compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProb = netMHC_II_EL_pep15.groupby('sequence name')['mhcii-np cleavage probability score'].mean().reset_index().rename(columns={'processing total score': 'netMHC_II_pep15_preProcess_cleavProb'})

# mhcii-np cleavage probability percentile rank

# remove the rows with the cleavage probability percentile rank of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] != '-']
# make the column with the cleavage probability percentile rank into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'])
# compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProbPercentile = netMHC_II_EL_pep15.groupby('sequence name')['mhcii-np cleavage probability percentile rank'].mean().reset_index().rename(columns={'mhcii-np cleavage probability percentile rank': 'netMHC_II_pep15_preProcess_cleavProbPercentile'})

# rename the column '...' to nr_aggs
waltz = waltz.rename(columns={'...': 'nr_aggs'})

# compute the total number of amio acids that are considerd immunogenetic
def sum_ranges(s):
    if pd.isna(s):
        return 0
    total = 0
    for part in s.split(';'):
        start, end = part.strip().split('-')
        total += int(end) - int(start) + 1 # beacuse the values are inclusive
    return total

waltz['nr_aggs'] = waltz['waltz_score'].apply(sum_ranges)

In [ ]:
# Merge scores from all predictors into one df

# create df with all the dfs that shall me merged into one
dfs_to_merge = [
    netMHC1_pep9_percentile,
    netMHC1_pep9_immunogenicity_score,
    netMCH1_pep9_preProcess,
    netMHC1_pep14_percentile,
    netMHC1_pep14_immunogenicity_score,
    netMCH1_pep14_preProcess,
    netMHC_II_pep12_percentile,
    netMHC_II_pep12_immunogenicity_score,
    netMHC_II_pep15_percentile,
    netMHC_II_pep15_immunogenicity_score,
    netMHC_II_pep15_preProcess_cleavProb,
    netMHC_II_pep15_preProcess_cleavProbPercentile,
]

# Rename the sequence name column to antibody to make the merging easier. 
dfs_to_merge = [df.rename(columns={'sequence name': 'antibody'}) for df in dfs_to_merge]

# Create the df with all scores, start with the ADA scores
all_predictors_andADA = ADA

# Merge all dfs on antibody name
for df in dfs_to_merge:
    all_predictors_andADA = all_predictors_andADA.merge(df, on='antibody', how='left')

# remove all whitespaces in antobdy column
all_predictors_andADA['antibody'] = all_predictors_andADA['antibody'].str.strip()

# Then merge the three last dfs
all_predictors_andADA = all_predictors_andADA.merge(waltz[['antibody','nr_aggs']], on='antibody', how='left').rename(columns={'nr_aggs': 'waltz_nr_aggs'})
all_predictors_andADA = all_predictors_andADA.merge(biophi_KabKabRelaxed, on='antibody', how='left')
all_predictors_andADA = all_predictors_andADA.merge(biophi_KabKabStrict, on='antibody', how='left')

all_predictors_andADA

In [ ]:
# Make all white space and '-' in the column names to '_'
all_predictors_andADA.columns = (
    all_predictors_andADA.columns
    .str.replace(r'[\s\-]+', '_', regex=True)
)

In [ ]:
# export score table
all_predictors_andADA.to_csv("all_predictors_217AB(biophidata).csv")

# Scatterplot ADA against all 15 predictors

In [ ]:
# for loop for all columns, make a scatter plot for each predictor against ADA

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()

for i in range(2, 17):
    ax = axes[i - 2]
    
    y = all_predictors_andADA['ADA_percentage']
    x = all_predictors_andADA.iloc[:, i]
    
    ax.scatter(x, y)
    ax.set_title(all_predictors_andADA.columns[i])
    ax.set_ylabel('ADA percantage')

plt.tight_layout()
plt.show()

# Correlation matrix

In [ ]:
# First create a df without the antibody name and ADA percantage
ADA_corrtest = all_predictors_andADA.drop(columns=['antibody'])
pearson_corr = ADA_corrtest.corr(method='pearson')

plt.figure(figsize=(12, 10))
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm')
plt.title("Pearson Correlation Heatmap")
plt.show()

In [ ]:
# print the correlation of the ADA percentage column
pearson_corr['ADA_percentage'].sort_values(ascending=False)

# Lasso regression

In [ ]:
# Create feature variables
X = all_predictors_andADA.drop(columns=['antibody', 'ADA_percentage']) # all except the response variable and the antibody names
y = all_predictors_andADA['ADA_percentage'] # the response varibale

model = make_pipeline(
    StandardScaler(),
    LassoCV(cv=5,max_iter=10000)  # cross-validation
)

model.fit(X, y)

# coefficients
coef = model.named_steps['lassocv'].coef_

selected_features = X.columns[coef != 0]

# print the features that Lasso "decided" are the best once to use for the prediction
selected_features

# Manual building models
This is to also look at other combinations that what Lasso generated and check how much they influence the results. Models with biology in mind for example.

In [ ]:
# print all column names, to get an overview of what potetnial predictors can be used 
X.iloc[1]

In [ ]:
# model 1
# The two features considred best my Lasso
model1 = sm.ols(formula= 'ADA_percentage ~ netMHC_II_pep12_percentile + netMHC_II_pep15_immunogenicity_score', 
                data=all_predictors_andADA).fit()

# Compute and print model fit calculations
print('AIC, R^2, adjusted R^2')
print("Model1:", model1.aic, model1.rsquared, model1.rsquared_adj)

# Visualization of results